In [ ]:
import os
import time
import math
import pandas as pd
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

import igraph as ig
import leidenalg as la
from sklearn.metrics import adjusted_rand_score
from itertools import combinations
import pickle

In [ ]:
import senepy as sp
import random

In [ ]:
pd.set_option("display.max_rows", 100)

In [ ]:
adata = sc.read_h5ad("../adata_all_harm_ribo.h5ad")
adata.raw = adata.copy()

In [ ]:
with open(
    "../senepy_denovo_signatures_code/ready/human_hubs.pkl",
    "rb",
) as file:
    calc_hubs = pickle.load(file)

with open(
    "../senepy_denovo_signatures_code/ready/human_signatures.pkl",
    "rb",
) as file:
    calc_sigs = pickle.load(file)

#### Calc senepy scores (signatures)

In [16]:
score_colname = "sp_score_sigs"
adata.obs[score_colname] = np.nan

for (t, c), hub in tqdm(calc_sigs.items()):
    ct_mask = (adata.obs.Tissue_global == t) & (adata.obs.cell_type_final == c)
    scores = sp.score_all_cells(
        adata[ct_mask], hub=hub, identifiers=["GSE_id"], binarize=True
    )
    adata.obs.loc[ct_mask, score_colname] = scores

  0%|                                                                                            | 0/78 [00:00<?, ?it/s]

100%|███████████████████████████████████████████████████████████████████████████████████| 78/78 [23:21<00:00, 17.97s/it]


In [ ]:
obs_filt = adata.obs.rename(columns={score_colname: "sp"})["sp"]

In [27]:
obs_filt

AAACCTGAGGCCGAAT-1-GSM3758115                               14.166038
AAACCTGAGTTAAGTG-1-GSM3758115                                2.127291
AAACCTGCAGTCAGAG-1-GSM3758115                                     NaN
AAACCTGGTCTTCTCG-1-GSM3758115                                1.607762
AAACCTGGTGCAACTT-1-GSM3758115                               11.074797
                                                              ...    
HCAHeartST13180619_HCAHeartST13177116_GCTCACAAGGCGCACT-1     0.942527
HCAHeartST13180619_HCAHeartST13177116_GAGTCATTCTTGGACG-1     0.175402
HCAHeartST13180619_HCAHeartST13177116_GTACTAATCTTACTCG-1     1.207378
HCAHeartST13180619_HCAHeartST13177116_CTTGAATCAAACGGGC-1    -0.020000
HCAHeartST13180619_HCAHeartST13177116_AGGCTAGCAGAAATGC-1    -0.012048
Name: sp, Length: 1454732, dtype: float64

In [ ]:
obs_filt.to_csv(".././0.2_sp_scores_dw.csv")